In [162]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,sampleID,sentence1,sentence2,score
0,0,A plane is taking off.,An air plane is taking off.,5.00
1,1,A man is playing a large flute.,A man is playing a flute.,3.80
2,2,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncoo...,3.80
3,3,Three men are playing chess.,Two men are playing chess.,2.60
4,4,A man is playing the cello.,A man seated is playing the cello.,4.25


In [163]:
#subtasks

subtask1 = []
subtask2 = []
subtask3 = []

def subtasks(row):
    sent1 = row.sentence1
    sent2 = row.sentence2
    
    lungMedie = (len(sent1) + len(sent2)) / 2
    if lungMedie < 50:
        subtask1.append("Short")
    elif lungMedie < 100:
        subtask1.append("Medium")
    else:
        subtask1.append("Long")

    nr_cuv = len(sent1.split(" ")) + len(sent2.split(" "))
    subtask2.append(nr_cuv)
    
    dif_abs = abs(len(sent1) - len(sent2))
    subtask3.append(dif_abs)
    return row

df_test = df_test.apply(subtasks, axis=1)

print(subtask1[:5])
print(subtask2[:5])
print(subtask3[:5])

['Short', 'Short', 'Short', 'Short', 'Short']
[12, 19, 13, 13, 12]
[1, 8, 6, 3, 4]


In [164]:
df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.sampleID,
    "answer": subtask1
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.sampleID,
    "answer": subtask2
})

df_3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.sampleID,
    "answer": subtask3
})

df_subtasks = pd.concat([df_1, df_2, df_3])

df_subtasks

,subtaskID,datapointID,answer
0,1,5749,Short
1,1,5750,Short
2,1,5751,Short
3,1,5752,Short
4,1,5753,Short
...,...,...,...
1374,3,7123,14
1375,3,7124,8
1376,3,7125,17
1377,3,7126,0


In [165]:
#preprocessing
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', "", text)
    text = re.sub(r'[\d+]', "", text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return ' '.join(tokens)

preprocess(df_train.sentence1[2])

'man spread shreded cheese pizza'

In [166]:
df_train.sentence1 = df_train.sentence1.apply(preprocess)
df_train.sentence2 = df_train.sentence2.apply(preprocess)

df_test.sentence1 = df_test.sentence1.apply(preprocess)
df_test.sentence2 = df_test.sentence2.apply(preprocess)

df_train.head()

,sampleID,sentence1,sentence2,score
0,0,plane take,air plane take,5.00
1,1,man play large flute,man play flute,3.80
2,2,man spread shreded cheese pizza,man spread shredded cheese uncooked pizza,3.80
3,3,man play chess,man play chess,2.60
4,4,man play cello,man seat play cello,4.25


In [199]:
#vectorizing tfidf
from sklearn.feature_extraction.text import TfidfVectorizer

all_texts = pd.concat([df_train.sentence1, df_train.sentence2, df_test.sentence1, df_test.sentence2])
vect = TfidfVectorizer(ngram_range=(3,5), analyzer='char_wb')
vect.fit(all_texts)

X1 = vect.transform(df_train.sentence1)
X2 = vect.transform(df_train.sentence2)

X1_final = vect.transform(df_test.sentence1)
X2_final = vect.transform(df_test.sentence2)

X1

<5749x52150 sparse matrix of type '<class 'numpy.float64'>'
	with 470551 stored elements in Compressed Sparse Row format>

In [235]:
#extragem feature urile - cos similarity, nr de cuv, dif intre nrele de caractere, counturi si jaccard scris foarte urat

df_train['word_count_1'] = [len(row.sentence1.split(" ")) for _,row in df_train.iterrows()]
df_train['word_count_2'] = [len(row.sentence2.split(" ")) for _,row in df_train.iterrows()]
df_train['char_dif'] = [abs(len(row.sentence1) - len(row.sentence2)) for _,row in df_train.iterrows()]
df_train['char_count'] = [len(row.sentence1) + len(row.sentence2) for _,row in df_train.iterrows()]
df_train['jaccard'] = [len(set(row.sentence1.split()) & set(row.sentence2.split())) / len(set(row.sentence1.split()) | set(row.sentence2.split())) for _,row in df_train.iterrows()]

df_test['word_count_1'] = [len(row.sentence1.split(" ")) for _,row in df_test.iterrows()]
df_test['word_count_2'] = [len(row.sentence2.split(" ")) for _,row in df_test.iterrows()]
df_test['char_dif'] = [abs(len(row.sentence1) - len(row.sentence2)) for _,row in df_test.iterrows()]
df_test['char_count'] = [len(row.sentence1) + len(row.sentence2) for _,row in df_test.iterrows()]
df_test['jaccard'] = [len(set(row.sentence1.split()) & set(row.sentence2.split())) / len(set(row.sentence1.split()) | set(row.sentence2.split())) if len(set(row.sentence1.split()) | set(row.sentence2.split()))>0 else 0 for _,row in df_test.iterrows()]

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

df_train['cosine_similarity'] = np.diag(cosine_similarity(X1, X2))
df_test['cosine_similarity'] = np.diag(cosine_similarity(X1_final, X2_final))

df_train.head()

,sampleID,sentence1,sentence2,score,word_count_1,word_count_2,char_dif,char_count,cosine_similarity,manhattan_dist,jaccard,length_ratio
0,0,plane take,air plane take,5.00,2,3,4,24,0.874033,1.753703,0.666667,0.714286
1,1,man play large flute,man play flute,3.80,4,3,6,34,0.818692,2.864610,0.750000,1.428571
2,2,man spread shreded cheese pizza,man spread shredded cheese uncooked pizza,3.80,5,6,10,72,0.679034,5.856119,0.571429,0.756098
3,3,man play chess,man play chess,2.60,3,3,0,28,1.000000,0.000000,1.000000,1.000000
4,4,man play cello,man seat play cello,4.25,3,4,5,33,0.845396,2.298284,0.750000,0.736842


In [236]:
df_train.columns

Index(['sampleID', 'sentence1', 'sentence2', 'score', 'word_count_1',
       'word_count_2', 'char_dif', 'char_count', 'cosine_similarity',
       'manhattan_dist', 'jaccard', 'length_ratio'],
      dtype='object')

In [243]:
features = ['word_count_1', 'word_count_2', 'char_count', 'char_dif', 'cosine_similarity', 'jaccard']
target = 'score'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

In [244]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

model = CatBoostRegressor(random_state=42, silent=True)

cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')

array([-0.79820763, -0.84992878, -0.7357154 , -0.81257725, -0.8769623 ])

In [245]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [246]:
df_preds = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": df_test.sampleID,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,4,5749,2.648031
1,4,5750,3.562413
2,4,5751,4.264107
3,4,5752,3.473456
4,4,5753,2.367551


In [247]:
submission = pd.concat([df_subtasks, df_preds])

submission

,subtaskID,datapointID,answer
0,1,5749,Short
1,1,5750,Short
2,1,5751,Short
3,1,5752,Short
4,1,5753,Short
...,...,...,...
1374,4,7123,1.340918
1375,4,7124,2.211491
1376,4,7125,2.083992
1377,4,7126,0.928782


In [248]:
submission.to_csv("submission.csv", index=False)